In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [1]:
import numpy as np
import cv2

# Camera Calibration Parameters

# intinsic matrices
K1 = np.array([
    [912.38344, 0, 712.36724],
    [0, 914.31159, 308.10289],
    [0, 0, 1]
], dtype=np.float64)

K2 = np.array([
    [918.72766, 0, 713.49119],
    [0, 922.79699, 312.35957],
    [0, 0, 1]
], dtype=np.float64)

# Distortion Coefficients (D): Radial and Tangential
D1 = np.array([0.09993326, -0.16873164, 0.0, 0.0, 0.0], dtype=np.float64)
D2 = np.array([0.02403417, -0.04766215, 0.0, 0.0, 0.0], dtype=np.float64)

# Stereo Extrinsic (R and T)
R = np.array([
    [0.99999374, 0.00281180, 0.00214595],
    [-0.00278851, 0.99993801, -0.01077938],
    [-0.00217613, 0.01077333, 0.99993960]
], dtype=np.float64)

T = np.array([
    [-1410.176887], # X (Baseline)
    [-11.138618],   # Y
    [-9.678103]     # Z
], dtype=np.float64)

In [ ]:
import cv2
import numpy as np

""" Code to obtain image coordinates from clicking on the right locations on the image"""

IMAGE_PATH = 'stereo_cam0_2c.jpg'
WINDOW_NAME = 'Click to get Coordinates'
clicked_points = []

def get_pixel_coordinates(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        coords = (x, y)
        clicked_points.append(coords)
        
        print(f"Clicked Coordinates (u, v): {coords}")
        
        img_display = param['img'].copy()
        for pt in clicked_points:
            cv2.circle(img_display, pt, 5, (0, 255, 0), -1)

        cv2.imshow(WINDOW_NAME, img_display)

def run_picker():
    try:
        img = cv2.imread(IMAGE_PATH)
        if img is None:
            raise FileNotFoundError(f"Could not load image at path: {IMAGE_PATH}")
    except Exception as e:
        print(f"Error loading image: {e}")
        return

    cv2.namedWindow(WINDOW_NAME)
    cv2.setMouseCallback(WINDOW_NAME, get_pixel_coordinates, {'img': img})

    print("\n--- Image Pixel Picker Started ---")
    print(f"Loading: {IMAGE_PATH}")
    print("Instructions: Click on the image to get pixel coordinates.")
    print("Press the 'ESC' key to close the window and see the final list of points.")
    
    while True:
        cv2.imshow(WINDOW_NAME, img)
        key = cv2.waitKey(20) & 0xFF
        if key == 27:
            break

    cv2.destroyAllWindows()

    print("\n--- Final List of Clicked Coordinates (u, v) ---")
    for i, pt in enumerate(clicked_points):
        print(f"Point {i+1}: {pt}")


if __name__ == '__main__':
    run_picker()


--- Image Pixel Picker Started ---
Loading: stereo_cam0_2c.jpg
Instructions: Click on the image to get pixel coordinates.
Press the 'ESC' key to close the window and see the final list of points.
Clicked Coordinates (u, v): (38, 86)
Clicked Coordinates (u, v): (24, 106)
Clicked Coordinates (u, v): (11, 211)
Clicked Coordinates (u, v): (22, 262)

--- Final List of Clicked Coordinates (u, v) ---
Point 1: (38, 86)
Point 2: (24, 106)
Point 3: (11, 211)
Point 4: (22, 262)

NumPy Array Format (for copying into your triangulation script):
[[ 38  86]
 [ 24 106]
 [ 11 211]
 [ 22 262]]


In [ ]:
def triangulate_points(K1, D1, K2, D2, R, T, points_L_px, points_R_px):
    """
    Undistorts pixel points and then performs linear triangulation.
    """
    # convert distorted pixels to normalized coordinates
    points_L = points_L_px.reshape(-1, 1, 2).astype(np.float64)
    points_R = points_R_px.reshape(-1, 1, 2).astype(np.float64)

    undistorted_L = cv2.undistortPoints(points_L, K1, D1, P=K1).squeeze()
    undistorted_R = cv2.undistortPoints(points_R, K2, D2, P=K2).squeeze()
    
    # define projection matrices
    P1 = K1 @ np.hstack((np.eye(3), np.zeros((3, 1))))
    
    R_T = np.hstack((R, T))
    P2 = K2 @ R_T

    # triangulation
    points_4d = cv2.triangulatePoints(P1, P2, undistorted_L.T, undistorted_R.T)

    # homogeneous to Euclidean
    points_3d = (points_4d / points_4d[3, :]).T[:, :3]

    return points_3d


# Globe Point
globe_point_L = np.array([[1097,  139]]) 
globe_point_R = np.array([[1044,  125]])

# Ground Plane Points
ground_points_L = np.array([
    [104,  98],
 [101 ,118],
 [106 ,221],
 [114 ,273]
])
ground_points_R = np.array([
    [ 38 , 86],
 [ 24 ,106],
 [ 11 ,211],
 [ 22 ,262]
])

print("\n--- Manual Pixel Points Defined (u, v) ---")
print(f"Globe Point Left: {globe_point_L.flatten()}")
print(f"Ground Points Left (Sample): {ground_points_L[0].flatten()}\n")


--- Manual Pixel Points Defined (u, v) ---
Globe Point Left: [1097  139]
Ground Points Left (Sample): [104  98]



In [ ]:
# Triangulate the Globe Point
P_globe_3d = triangulate_points(K1, D1, K2, D2, R, T, globe_point_L, globe_point_R)
print("\n--- Globe Point Triangulation Result (P_globe) ---")
print(f"3D Coordinates (X, Y, Z) in mm: {P_globe_3d[0]}")
Xg, Yg, Zg = P_globe_3d[0]


# Triangulate the Ground Plane Points
P_ground_3d = triangulate_points(K1, D1, K2, D2, R, T, ground_points_L, ground_points_R)
print("\n--- Ground Plane Points Triangulation Result (P_ground) ---")
print(f"Number of ground points triangulated: {len(P_ground_3d)}")


# center the points
centroid = np.mean(P_ground_3d, axis=0)
A = P_ground_3d - centroid

# SVD to obtain the normal vector
U, S, Vh = np.linalg.svd(A)
normal_vector = Vh[-1, :]

# Define A, B, C from the normal vector
A_fit, B_fit, C_fit = normal_vector

# Calculate D using the centroid: Ax + By + Cz + D = 0 => D = -(Ax + By + Cz)
D_fit = -np.dot(normal_vector, centroid)

print("\n--- Ground Plane Fit (Ax + By + Cz + D = 0) ---")
print(f"Normal Vector (A, B, C): {A_fit:.4f}, {B_fit:.4f}, {C_fit:.4f}")
print(f"D: {D_fit:.4f}")

# Height is the distance from the point P_globe to the fitted plane.
numerator = abs(A_fit * Xg + B_fit * Yg + C_fit * Zg + D_fit)
denominator = np.sqrt(A_fit**2 + B_fit**2 + C_fit**2)

globe_height_mm = numerator / denominator


print(f"Calculated Globe Height above Ground Plane:")
print(f"{globe_height_mm:.2f} mm")

--- Undistortion and Normalization Complete ---

--- Globe Point Triangulation Result (P_globe) ---
3D Coordinates (X, Y, Z) in mm: [ 9758.04676669 -4362.31389451 23455.03713444]
--- Undistortion and Normalization Complete ---

--- Ground Plane Points Triangulation Result (P_ground) ---
Number of ground points triangulated: 4

--- Ground Plane Fit (Ax + By + Cz + D = 0) ---
Normal Vector (A, B, C): -0.8539, 0.1265, -0.5048
D: -562.4024

**Calculated Globe Height above Ground Plane:**
**21286.10 mm**
